## Example 1: Basic Content Extraction

In [ ]:
# Example 1: Basic content extraction
print("Example 1: Basic content extraction")

from readability import Document

# Sample HTML with navigation and main content
html = """
<html><head><title>Machine Learning Guide</title></head>
<body>
    <nav><a href="/">Home</a></nav>
    <article>
        <h1>Introduction to Machine Learning</h1>
        <p>Machine learning is transforming technology through data-driven insights.</p>
        <p>This guide covers fundamental concepts and practical applications.</p>
    </article>
    <footer>Copyright 2023</footer>
</body></html>
"""

doc = Document(html)
print(f"✓ Title: {doc.title()}")
print(f"✓ Short title: {doc.short_title()}")

summary = doc.summary()
print(f"✓ Summary length: {len(summary)} chars")
print(f"✓ Content preview: {summary[:100]}...")

from lxml import html as lxml_html
summary_text = ' '.join(lxml_html.fromstring(summary).text_content().split())
assert doc.title() == 'Machine Learning Guide' and doc.short_title() == 'Machine Learning Guide'
assert 'Machine learning is transforming technology through data-driven insights.' in summary_text, summary_text
assert 'Copyright 2023' not in summary_text, summary_text  # the footer is dropped

## Example 2: News article extraction

In [ ]:
# Example 2: News article extraction
print("Example 2: News article extraction")

from readability import Document
from lxml import html as lxml_html

article_html = """
<html><head><title>AI Breakthrough Announced</title></head>
<body>
    <div class="ads">Advertisement</div>
    <div class="main">
        <h1>Major AI Breakthrough in Healthcare</h1>
        <p>Researchers announced a breakthrough in medical AI today.</p>
        <p>The new system improves diagnostic accuracy by 25%.</p>
    </div>
    <aside>Related articles</aside>
</body></html>
"""

doc = Document(article_html)
print(f"✓ Extracted title: {doc.title()}")

# Get clean HTML content
clean_content = doc.summary()
tree = lxml_html.fromstring(clean_content)
text = tree.text_content()
print(f"✓ Text content: {text[:150]}...")

clean_text = ' '.join(text.split())
assert doc.title() == 'AI Breakthrough Announced'
assert 'The new system improves diagnostic accuracy by 25%.' in clean_text, clean_text
# The advertisement and the aside are not part of the extracted article
assert 'Advertisement' not in clean_text and 'Related articles' not in clean_text, clean_text

## Example 3: Blog post processing

In [ ]:
# Example 3: Blog post processing
print("Example 3: Blog post processing")

from readability import Document
from lxml import html as lxml_html

blog_html = """
<html><head><title>Python Tips and Tricks</title></head>
<body>
    <header><nav>Menu</nav></header>
    <article>
        <h1>10 Python Tips for Better Code</h1>
        <p>Python offers elegant solutions to common problems.</p>
        <h2>List Comprehensions</h2>
        <p>Use list comprehensions for cleaner, faster code.</p>
        <h2>Context Managers</h2>
        <p>Always use context managers for resource management.</p>
    </article>
    <div class="comments">Comments section</div>
</body></html>
"""

doc = Document(blog_html)
summary = doc.summary()
tree = lxml_html.fromstring(summary)

# Count preserved elements
headers = tree.xpath('.//h1 | .//h2')
paragraphs = tree.xpath('.//p')
print(f"✓ Headers preserved: {len(headers)}")
print(f"✓ Paragraphs extracted: {len(paragraphs)}")

assert [h.text_content() for h in headers] == ['10 Python Tips for Better Code', 'List Comprehensions', 'Context Managers']
assert len(paragraphs) == 3, len(paragraphs)
assert 'Comments section' not in tree.text_content()

## Example 4: Multiple document processing

In [ ]:
# Example 4: Multiple document processing
print("Example 4: Multiple document processing")

from readability import Document

documents = {
    "tech": "<html><head><title>Tech News</title></head><body><article><h1>New Framework Released</h1><p>A new web framework was released today.</p></article></body></html>",
    "science": "<html><head><title>Science Update</title></head><body><article><h1>Climate Study</h1><p>New research reveals climate patterns.</p></article></body></html>",
    "business": "<html><head><title>Market Report</title></head><body><article><h1>Stock Market Update</h1><p>Markets showed gains today.</p></article></body></html>"
}

results = {}
for name, html in documents.items():
    doc = Document(html)
    results[name] = {
        'title': doc.title(),
        'content_length': len(doc.summary())
    }
    print(f"✓ {name}: {results[name]['title']}, {results[name]['content_length']} chars")

avg_length = sum(r['content_length'] for r in results.values()) / len(results)
print(f"✓ Average content length: {avg_length:.0f} chars")

assert {name: r['title'] for name, r in results.items()} == {
    'tech': 'Tech News', 'science': 'Science Update', 'business': 'Market Report'}
expected_text = {
    'tech': 'A new web framework was released today.',
    'science': 'New research reveals climate patterns.',
    'business': 'Markets showed gains today.',
}
for name, html in documents.items():
    assert expected_text[name] in Document(html).summary(), name

## Example 5: Content quality analysis

In [ ]:
# Example 5: Content quality analysis
print("Example 5: Content quality analysis")

from readability import Document
from lxml import html as lxml_html

complex_html = """
<html><head><title>Web Development Best Practices</title></head>
<body>
    <nav>Navigation</nav>
    <aside>Sidebar ads</aside>
    <main>
        <h1>Web Development Best Practices</h1>
        <p>Modern web development requires attention to performance and accessibility.</p>
        <h2>Performance Optimization</h2>
        <p>Optimize images and use lazy loading for better performance.</p>
        <h2>Accessibility</h2>
        <p>Ensure your site is accessible to all users with proper ARIA labels.</p>
    </main>
    <footer>Footer content</footer>
</body></html>
"""

doc = Document(complex_html)
summary = doc.summary()
tree = lxml_html.fromstring(summary)
text = tree.text_content()

print(f"✓ Title: {doc.short_title()}")
print(f"✓ Content: {len(text)} chars, {len(text.split())} words")
print(f"✓ Structure: {len(tree.xpath('.//h2'))} sections")

assert doc.short_title() == 'Web Development Best Practices'
assert len(tree.xpath('.//h2')) == 2
assert 'proper ARIA labels' in text
assert 'Sidebar ads' not in text and 'Footer content' not in text